# Time series: practice notebook

4 exercises, each with hidden tests. Data is loaded as **polars**; pandas only appears where a library insists on it.

**How to use it**
1. Run the three setup cells below once (Runtime → Run all works too; the exercise cells just print `Wrong answer` until you solve them).
2. For each exercise, write your code in the solution cell, run it, then run the `check(...)` cell under it. It prints **Accepted** or which test failed and why.
3. Stuck? `solution("t1")` prints the reference solution.

Opened from the roadmap it runs in Google Colab. To keep your work, use File → Save a copy in Drive the first time.

In [ ]:
# Install anything missing (Colab already has numpy, scipy, statsmodels, scikit-learn, matplotlib)
import importlib.util, subprocess, sys
for _p in ['polars', 'arch']:
    if importlib.util.find_spec(_p) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _p], check=False)


## Setup: data
The data every exercise uses. Read it if you like; don't change it.

In [ ]:
import warnings
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, acf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.arima_process import ArmaProcess
warnings.filterwarnings("ignore")

def rdata(name, package):
    """Load an R dataset (via statsmodels / Rdatasets) as a polars DataFrame, NaN → null."""
    df = pl.from_pandas(sm.datasets.get_rdataset(name, package, cache=True).data.reset_index(drop=True)).drop("rownames", strict=False)
    return df.with_columns(pl.col(pl.Float64).fill_nan(None))

eu = rdata("EuStockMarkets", "datasets")   # daily closes 1991-1998: DAX, SMI, CAC, FTSE
dax = eu["DAX"].to_numpy().astype(float)
arma_x = ArmaProcess(ar=[1, -0.6], ma=[1, 0.4]).generate_sample(600, distrvs=np.random.default_rng(4).standard_normal)

In [ ]:
# Test runner: check(id) and solution(id)
import base64 as _b64, json as _json, io as _io, contextlib as _ctx, traceback as _tb
_LAB = _json.loads(_b64.b64decode("eyJ0MSI6IHsic29sdXRpb24iOiAiZGVmIGFkZl9wdmFsdWVzKHByaWNlcyk6XG4gICAgciA9IG5wLmRpZmYobnAubG9nKHByaWNlcykpXG4gICAgcmV0dXJuIGZsb2F0KGFkZnVsbGVyKHByaWNlcylbMV0pLCBmbG9hdChhZGZ1bGxlcihyKVsxXSlcbiIsICJ0ZXN0cyI6ICJcbmRlZiBjbG9zZShhLCBiLCByZWw9MWUtNiwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYSA9IGZsb2F0KGEpOyBiID0gZmxvYXQoYilcbiAgICBhc3NlcnQgYWJzKGEgLSBiKSA8PSByZWwgKiBtYXgoMS4wLCBhYnMoYikpLCBmXCJ7d2hhdH0gPSB7YTouNmd9LCBleHBlY3RlZCB7YjouNmd9XCJcbmRlZiBiZXR3ZWVuKGEsIGxvLCBoaSwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYXNzZXJ0IGxvIDw9IGZsb2F0KGEpIDw9IGhpLCBmXCJ7d2hhdH0gPSB7ZmxvYXQoYSk6LjRmfSwgZXhwZWN0ZWQgYmV0d2VlbiB7bG99IGFuZCB7aGl9XCJcbmRlZiBzYW1lX2tleXMoYSwgYiwgd2hhdD1cInJlc3VsdFwiKTpcbiAgICBhc3NlcnQgaXNpbnN0YW5jZShhLCBkaWN0KSwgZlwie3doYXR9IHNob3VsZCBiZSBhIGRpY3QsIGdvdCB7dHlwZShhKS5fX25hbWVfX31cIlxuICAgIGFzc2VydCBzZXQoYSkgPT0gc2V0KGIpLCBmXCJ7d2hhdH0ga2V5cyB7c29ydGVkKGEpfSwgZXhwZWN0ZWQge3NvcnRlZChiKX1cIlxuXG5kZWYgX3QoeCk6XG4gICAgYSwgYiA9IFVbXCJhZGZfcHZhbHVlc1wiXSh4KSwgUltcImFkZl9wdmFsdWVzXCJdKHgpXG4gICAgY2xvc2UoYVswXSwgYlswXSwgMWUtNiwgXCJwcmljZSBwLXZhbHVlXCIpOyBjbG9zZShhWzFdLCBiWzFdLCAxZS02LCBcInJldHVybiBwLXZhbHVlXCIpXG5URVNUUyA9IFsoXCJEQVhcIiwgbGFtYmRhOiBfdChkYXgpKSwgKFwiaGlkZGVuOiBGVFNFXCIsIGxhbWJkYTogX3QoZXVbXCJGVFNFXCJdLnRvX251bXB5KCkuYXN0eXBlKGZsb2F0KSkpXVxuIiwgImxhbmciOiAicHkifSwgInQyIjogeyJzb2x1dGlvbiI6ICJkZWYgYXIxX2ZpdChwaGksIG49MTAwMCwgc2VlZD0wKTpcbiAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoc2VlZClcbiAgICBlcHMgPSBybmcubm9ybWFsKHNpemU9bilcbiAgICB4ID0gbnAuZW1wdHkobik7IHhbMF0gPSBlcHNbMF1cbiAgICBmb3IgdCBpbiByYW5nZSgxLCBuKTpcbiAgICAgICAgeFt0XSA9IHBoaSAqIHhbdCAtIDFdICsgZXBzW3RdXG4gICAgcmV0dXJuIGZsb2F0KEFSSU1BKHgsIG9yZGVyPSgxLCAwLCAwKSwgdHJlbmQ9XCJuXCIpLmZpdCgpLnBhcmFtc1swXSlcbiIsICJ0ZXN0cyI6ICJcbmRlZiBjbG9zZShhLCBiLCByZWw9MWUtNiwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYSA9IGZsb2F0KGEpOyBiID0gZmxvYXQoYilcbiAgICBhc3NlcnQgYWJzKGEgLSBiKSA8PSByZWwgKiBtYXgoMS4wLCBhYnMoYikpLCBmXCJ7d2hhdH0gPSB7YTouNmd9LCBleHBlY3RlZCB7YjouNmd9XCJcbmRlZiBiZXR3ZWVuKGEsIGxvLCBoaSwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYXNzZXJ0IGxvIDw9IGZsb2F0KGEpIDw9IGhpLCBmXCJ7d2hhdH0gPSB7ZmxvYXQoYSk6LjRmfSwgZXhwZWN0ZWQgYmV0d2VlbiB7bG99IGFuZCB7aGl9XCJcbmRlZiBzYW1lX2tleXMoYSwgYiwgd2hhdD1cInJlc3VsdFwiKTpcbiAgICBhc3NlcnQgaXNpbnN0YW5jZShhLCBkaWN0KSwgZlwie3doYXR9IHNob3VsZCBiZSBhIGRpY3QsIGdvdCB7dHlwZShhKS5fX25hbWVfX31cIlxuICAgIGFzc2VydCBzZXQoYSkgPT0gc2V0KGIpLCBmXCJ7d2hhdH0ga2V5cyB7c29ydGVkKGEpfSwgZXhwZWN0ZWQge3NvcnRlZChiKX1cIlxuXG5kZWYgX3QocGhpLCBzZWVkKTpcbiAgICBjbG9zZShVW1wiYXIxX2ZpdFwiXShwaGksIHNlZWQ9c2VlZCksIFJbXCJhcjFfZml0XCJdKHBoaSwgc2VlZD1zZWVkKSwgMWUtNCwgZlwiXHUwM2M2IGVzdGltYXRlIGZvciBcdTAzYzYgPSB7cGhpfVwiKVxuVEVTVFMgPSBbKFwiXHUwM2M2ID0gMC41XCIsIGxhbWJkYTogX3QoMC41LCAwKSksIChcImhpZGRlbjogXHUwM2M2ID0gLTAuN1wiLCBsYW1iZGE6IF90KC0wLjcsIDIpKV1cbiIsICJsYW5nIjogInB5In0sICJ0MyI6IHsic29sdXRpb24iOiAiZGVmIGJlc3RfYXJtYSh4LCBtYXhfcD0yLCBtYXhfcT0yKTpcbiAgICBiZXN0ID0gTm9uZVxuICAgIGZvciBwIGluIHJhbmdlKG1heF9wICsgMSk6XG4gICAgICAgIGZvciBxIGluIHJhbmdlKG1heF9xICsgMSk6XG4gICAgICAgICAgICBhaWMgPSBBUklNQSh4LCBvcmRlcj0ocCwgMCwgcSkpLmZpdCgpLmFpY1xuICAgICAgICAgICAgaWYgYmVzdCBpcyBOb25lIG9yIGFpYyA8IGJlc3RbMF06XG4gICAgICAgICAgICAgICAgYmVzdCA9IChhaWMsIHAsIHEpXG4gICAgcmV0dXJuIGJlc3RbMV0sIGJlc3RbMl1cbiIsICJ0ZXN0cyI6ICJcbmRlZiBjbG9zZShhLCBiLCByZWw9MWUtNiwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYSA9IGZsb2F0KGEpOyBiID0gZmxvYXQoYilcbiAgICBhc3NlcnQgYWJzKGEgLSBiKSA8PSByZWwgKiBtYXgoMS4wLCBhYnMoYikpLCBmXCJ7d2hhdH0gPSB7YTouNmd9LCBleHBlY3RlZCB7YjouNmd9XCJcbmRlZiBiZXR3ZWVuKGEsIGxvLCBoaSwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYXNzZXJ0IGxvIDw9IGZsb2F0KGEpIDw9IGhpLCBmXCJ7d2hhdH0gPSB7ZmxvYXQoYSk6LjRmfSwgZXhwZWN0ZWQgYmV0d2VlbiB7bG99IGFuZCB7aGl9XCJcbmRlZiBzYW1lX2tleXMoYSwgYiwgd2hhdD1cInJlc3VsdFwiKTpcbiAgICBhc3NlcnQgaXNpbnN0YW5jZShhLCBkaWN0KSwgZlwie3doYXR9IHNob3VsZCBiZSBhIGRpY3QsIGdvdCB7dHlwZShhKS5fX25hbWVfX31cIlxuICAgIGFzc2VydCBzZXQoYSkgPT0gc2V0KGIpLCBmXCJ7d2hhdH0ga2V5cyB7c29ydGVkKGEpfSwgZXhwZWN0ZWQge3NvcnRlZChiKX1cIlxuXG5kZWYgX3QoeCk6XG4gICAgYSwgYiA9IHR1cGxlKFVbXCJiZXN0X2FybWFcIl0oeCkpLCB0dXBsZShSW1wiYmVzdF9hcm1hXCJdKHgpKVxuICAgIGFzc2VydCBhID09IGIsIGZcIm9yZGVyIHthfSwgZXhwZWN0ZWQge2J9XCJcblRFU1RTID0gWyhcImFybWFfeFwiLCBsYW1iZGE6IF90KGFybWFfeCkpLFxuICAgICAgICAgKFwiaGlkZGVuOiBhbiBBUigyKVwiLCBsYW1iZGE6IF90KEFybWFQcm9jZXNzKGFyPVsxLCAtMC41LCAwLjNdKS5nZW5lcmF0ZV9zYW1wbGUoNTAwLCBkaXN0cnZzPW5wLnJhbmRvbS5kZWZhdWx0X3JuZyg4KS5zdGFuZGFyZF9ub3JtYWwpKSldXG4iLCAibGFuZyI6ICJweSJ9LCAidDQiOiB7InNvbHV0aW9uIjogImZyb20gYXJjaCBpbXBvcnQgYXJjaF9tb2RlbFxuXG5kZWYgYWNmX3NxKHIsIGxhZ3M9MTApOlxuICAgIHJldHVybiBhY2YociwgbmxhZ3M9bGFncylbMTpdLCBhY2YocioqMiwgbmxhZ3M9bGFncylbMTpdXG5cbmRlZiBnYXJjaF9maXQocik6XG4gICAgcmVzID0gYXJjaF9tb2RlbChyLCBtZWFuPVwiQ29uc3RhbnRcIiwgdm9sPVwiR0FSQ0hcIiwgcD0xLCBxPTEsIGRpc3Q9XCJub3JtYWxcIikuZml0KGRpc3A9XCJvZmZcIilcbiAgICBwID0gcmVzLnBhcmFtc1xuICAgIHJldHVybiB7XCJvbWVnYVwiOiBmbG9hdChwW1wib21lZ2FcIl0pLCBcImFscGhhXCI6IGZsb2F0KHBbXCJhbHBoYVsxXVwiXSksIFwiYmV0YVwiOiBmbG9hdChwW1wiYmV0YVsxXVwiXSksXG4gICAgICAgICAgICBcInBlcnNpc3RlbmNlXCI6IGZsb2F0KHBbXCJhbHBoYVsxXVwiXSArIHBbXCJiZXRhWzFdXCJdKX1cbiIsICJ0ZXN0cyI6ICJcbmRlZiBjbG9zZShhLCBiLCByZWw9MWUtNiwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYSA9IGZsb2F0KGEpOyBiID0gZmxvYXQoYilcbiAgICBhc3NlcnQgYWJzKGEgLSBiKSA8PSByZWwgKiBtYXgoMS4wLCBhYnMoYikpLCBmXCJ7d2hhdH0gPSB7YTouNmd9LCBleHBlY3RlZCB7YjouNmd9XCJcbmRlZiBiZXR3ZWVuKGEsIGxvLCBoaSwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYXNzZXJ0IGxvIDw9IGZsb2F0KGEpIDw9IGhpLCBmXCJ7d2hhdH0gPSB7ZmxvYXQoYSk6LjRmfSwgZXhwZWN0ZWQgYmV0d2VlbiB7bG99IGFuZCB7aGl9XCJcbmRlZiBzYW1lX2tleXMoYSwgYiwgd2hhdD1cInJlc3VsdFwiKTpcbiAgICBhc3NlcnQgaXNpbnN0YW5jZShhLCBkaWN0KSwgZlwie3doYXR9IHNob3VsZCBiZSBhIGRpY3QsIGdvdCB7dHlwZShhKS5fX25hbWVfX31cIlxuICAgIGFzc2VydCBzZXQoYSkgPT0gc2V0KGIpLCBmXCJ7d2hhdH0ga2V5cyB7c29ydGVkKGEpfSwgZXhwZWN0ZWQge3NvcnRlZChiKX1cIlxuXG5kZWYgX2FjZigpOlxuICAgIHIgPSAxMDAgKiBucC5kaWZmKG5wLmxvZyhkYXgpKVxuICAgIGEsIGIgPSBVW1wiYWNmX3NxXCJdKHIpLCBSW1wiYWNmX3NxXCJdKHIpXG4gICAgZm9yIGkgaW4gcmFuZ2UoMTApOlxuICAgICAgICBjbG9zZShhWzBdW2ldLCBiWzBdW2ldLCAxZS04LCBmXCJhY2Ygb2YgciBhdCBsYWcge2kgKyAxfVwiKTsgY2xvc2UoYVsxXVtpXSwgYlsxXVtpXSwgMWUtOCwgZlwiYWNmIG9mIHJcdTAwYjIgYXQgbGFnIHtpICsgMX1cIilcbmRlZiBfZyh4KTpcbiAgICByID0gMTAwICogbnAuZGlmZihucC5sb2coeCkpXG4gICAgYSwgYiA9IFVbXCJnYXJjaF9maXRcIl0ociksIFJbXCJnYXJjaF9maXRcIl0ocik7IHNhbWVfa2V5cyhhLCBiKVxuICAgIGZvciBrIGluIGI6IGNsb3NlKGFba10sIGJba10sIDJlLTMsIGspXG5URVNUUyA9IFsoXCJBQ0Ygb2YgcmV0dXJucyBhbmQgc3F1YXJlZCByZXR1cm5zXCIsIF9hY2YpLCAoXCJHQVJDSCBvbiBEQVhcIiwgbGFtYmRhOiBfZyhkYXgpKSxcbiAgICAgICAgIChcImhpZGRlbjogR0FSQ0ggb24gQ0FDXCIsIGxhbWJkYTogX2coZXVbXCJDQUNcIl0udG9fbnVtcHkoKS5hc3R5cGUoZmxvYXQpKSldXG4iLCAibGFuZyI6ICJweSJ9fQ==").decode())
_SETUP = _b64.b64decode("aW1wb3J0IHdhcm5pbmdzCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcG9sYXJzIGFzIHBsCmltcG9ydCBtYXRwbG90bGliLnB5cGxvdCBhcyBwbHQKaW1wb3J0IHN0YXRzbW9kZWxzLmFwaSBhcyBzbQpmcm9tIHN0YXRzbW9kZWxzLnRzYS5zdGF0dG9vbHMgaW1wb3J0IGFkZnVsbGVyLCBhY2YKZnJvbSBzdGF0c21vZGVscy50c2EuYXJpbWEubW9kZWwgaW1wb3J0IEFSSU1BCmZyb20gc3RhdHNtb2RlbHMudHNhLmFyaW1hX3Byb2Nlc3MgaW1wb3J0IEFybWFQcm9jZXNzCndhcm5pbmdzLmZpbHRlcndhcm5pbmdzKCJpZ25vcmUiKQoKZGVmIHJkYXRhKG5hbWUsIHBhY2thZ2UpOgogICAgIiIiTG9hZCBhbiBSIGRhdGFzZXQgKHZpYSBzdGF0c21vZGVscyAvIFJkYXRhc2V0cykgYXMgYSBwb2xhcnMgRGF0YUZyYW1lLCBOYU4g4oaSIG51bGwuIiIiCiAgICBkZiA9IHBsLmZyb21fcGFuZGFzKHNtLmRhdGFzZXRzLmdldF9yZGF0YXNldChuYW1lLCBwYWNrYWdlLCBjYWNoZT1UcnVlKS5kYXRhLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkpLmRyb3AoInJvd25hbWVzIiwgc3RyaWN0PUZhbHNlKQogICAgcmV0dXJuIGRmLndpdGhfY29sdW1ucyhwbC5jb2wocGwuRmxvYXQ2NCkuZmlsbF9uYW4oTm9uZSkpCgpldSA9IHJkYXRhKCJFdVN0b2NrTWFya2V0cyIsICJkYXRhc2V0cyIpICAgIyBkYWlseSBjbG9zZXMgMTk5MS0xOTk4OiBEQVgsIFNNSSwgQ0FDLCBGVFNFCmRheCA9IGV1WyJEQVgiXS50b19udW1weSgpLmFzdHlwZShmbG9hdCkKYXJtYV94ID0gQXJtYVByb2Nlc3MoYXI9WzEsIC0wLjZdLCBtYT1bMSwgMC40XSkuZ2VuZXJhdGVfc2FtcGxlKDYwMCwgZGlzdHJ2cz1ucC5yYW5kb20uZGVmYXVsdF9ybmcoNCkuc3RhbmRhcmRfbm9ybWFsKQ==").decode()

def _err(e):
    fr = [f for f in _tb.extract_tb(e.__traceback__) if f.filename.startswith("<ipython") or f.filename == "<string>"]
    return f"{type(e).__name__}: {e}"

def check(ex_id, query=None):
    """Run the hidden tests for one exercise, like Submit in the editor."""
    e = _LAB[ex_id]
    ref = {}
    with _ctx.redirect_stdout(_io.StringIO()):
        exec(_SETUP, ref)
        if e["lang"] == "sql":
            ref["__sql__"] = e["solution"]
        else:
            exec(e["solution"], ref)
    if e["lang"] == "sql":
        if query is None:
            print("Pass your query string: check(\"" + ex_id + "\", q)")
            return
        user = {"__sql__": query}
    else:
        user = dict(globals())
    tns = dict(ref)
    tns["U"], tns["R"] = user, ref
    exec(e["tests"], tns)
    ok, tests = 0, tns["TESTS"]
    for name, fn in tests:
        try:
            fn()
            print("  ✓ " + name)
            ok += 1
        except AssertionError as err:
            print("  ✗ " + name + ": " + (str(err) or "wrong answer"))
        except Exception as err:
            print("  ✗ " + name + ": " + _err(err))
    print(("Accepted" if ok == len(tests) else "Wrong answer") + f" · {ok} / {len(tests)} test cases passed")

def solution(ex_id):
    """Print the reference solution. Try once more first."""
    print(_LAB[ex_id]["solution"])

---
## 1. Prices vs returns: the ADF unit-root test
*Easy · id `t1`*

<p>Write <code>adf_pvalues(prices)</code> returning <code>(p_price, p_return)</code>: the <code>adfuller</code> p-values (default settings) for the price series and for its log returns <code>np.diff(np.log(prices))</code>.</p>
<p>Plot both with their ACFs. What does each p-value say, and why do quants model returns rather than prices?</p>

**Theory to know cold**
<ul><li>ADF H₀: a unit root (non-stationary). A large p-value for prices means you can't reject a random walk; a tiny one for returns means they look stationary.</li><li>The ACF of a random walk decays very slowly; regressions of one trending series on another give spurious R².</li></ul>

In [ ]:
def adf_pvalues(prices):
    # your code here
    pass

# Sample run
try:
    print(adf_pvalues(dax))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("t1")

---
## 2. Simulate an AR(1) and recover φ with statsmodels ARIMA
*Medium · id `t2`*

<p>Write <code>ar1_fit(phi, n=1000, seed=0)</code>: draw <code>eps = rng.normal(size=n)</code>, build x₀ = ε₀ and x_t = φx_{t−1} + ε_t with a loop, fit <code>ARIMA(x, order=(1, 0, 0), trend="n")</code>, and return the estimated φ.</p>
<p>Plot the sample ACF next to φᵏ. What happens to the estimate as φ → 1?</p>

**Theory to know cold**
<ul><li>AR(1) is stationary iff |φ| &lt; 1; its ACF is φᵏ and its PACF cuts off after lag 1. An MA(q) is the reverse: ACF cuts off after q.</li><li>Near φ = 1 the OLS/ML estimate is biased downward (Dickey-Fuller), which is why unit-root tests need special critical values.</li></ul>

In [ ]:
def ar1_fit(phi, n=1000, seed=0):
    rng = np.random.default_rng(seed)
    # your code here
    pass

# Sample run
try:
    print([round(ar1_fit(p), 3) for p in (0.2, 0.5, 0.9, 0.99)])
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("t2")

---
## 3. Pick an ARMA order by AIC
*Medium · id `t3`*

<p><code>arma_x</code> is a simulated ARMA series. Write <code>best_arma(x, max_p=2, max_q=2)</code>: fit <code>ARIMA(x, order=(p, 0, q))</code> (default trend) for every p ≤ max_p, q ≤ max_q and return the <code>(p, q)</code> with the lowest AIC.</p>
<p>Read the ACF and PACF first: could you have guessed the order?</p>

**Theory to know cold**
<ul><li>AIC = 2k − 2 log L rewards fit and penalises parameters; BIC penalises more and picks smaller models.</li><li>For forecasting, compare models on out-of-sample error, not only AIC.</li></ul>

In [ ]:
def best_arma(x, max_p=2, max_q=2):
    # your code here
    pass

# Sample run
try:
    print(best_arma(arma_x))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("t3")

---
## 4. GARCH(1,1) on DAX returns with the arch package
*Hard · id `t4`*

<p>Use percentage log returns r = 100 · diff(log DAX).</p>
<p>1. <code>acf_sq(r, lags=10)</code>: return <code>(acf of r, acf of r²)</code> at lags 1..lags (drop lag 0) using <code>acf(..., nlags=lags)</code>. Returns barely autocorrelate; squared returns do: volatility clustering.</p>
<p>2. <code>garch_fit(r)</code>: fit <code>arch_model(r, mean="Constant", vol="GARCH", p=1, q=1, dist="normal").fit(disp="off")</code> and return a dict <code>omega, alpha, beta, persistence</code> (α + β).</p>
<p>What is the long-run daily volatility ω/(1 − α − β), and the half-life of a volatility shock?</p>

**Theory to know cold**
<ul><li>σ²_t = ω + α ε²_{t−1} + β σ²_{t−1}. α measures how fast vol reacts to news, β how long it lingers; α + β close to 1 means persistent vol.</li><li>Half-life = log(0.5) / log(α + β) days.</li></ul>

In [ ]:
from arch import arch_model

def acf_sq(r, lags=10):
    # your code here
    pass

def garch_fit(r):
    # your code here
    pass

# Sample run
try:
    r = 100 * np.diff(np.log(dax))
    a, s = acf_sq(r); print(a.round(3)); print(s.round(3))
    print(garch_fit(r))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("t4")